In [0]:
%sql

SELECT COUNT(*) FROM bootcamp.silver.propiedades;
DESCRIBE TABLE bootcamp.silver.propiedades;

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_zona;

CREATE TABLE bootcamp.gold.dim_zona
(
zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
partido string,
region string,
ciudad string,
provincia string default "Buenos Aires",
pais string default "Argentina",
_created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT "Tabla Dimension Zonas - Star Schema";

insert into bootcamp.gold.dim_zona 
(partido,region,ciudad,provincia,pais)
select distinct
sp.partido,
sp.region,
case 
    when sp.region = "capital federal" then "CABA"
    else "GBA"
    end as ciudad,
"Buenos Aires" as provincia,
"Argentina" as pais
from bootcamp.silver.propiedades sp
WHERE sp.partido IS NOT NULL
ORDER BY sp.partido;

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_tipo_operacion;

CREATE TABLE bootcamp.gold.dim_tipo_operacion
(
tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
tipo_operacion string,
moneda string,
categoria string,
descripcion string,
_created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT "Tabla Dimension Tipo de Operacion - Star Schema";

insert into bootcamp.gold.dim_tipo_operacion 
(tipo_operacion,moneda,categoria,descripcion)
select distinct
tipo_operacion,
moneda,
case 
    when tipo_operacion = "alquiler" then "residencial"
    when tipo_operacion = "venta" then "residencial"
    when tipo_operacion = "alquiler_temporario" then "temporal"
    else "otro"
    end as categoria,
CASE 
    WHEN tipo_operacion = 'alquiler' THEN 'Alquiler residencial'
    WHEN tipo_operacion = 'venta' THEN 'Venta de propiedad'
    WHEN tipo_operacion = 'alquiler_temporario' THEN 'Alquiler temporal'
    ELSE 'Otro tipo'
END as descripcion
FROM bootcamp.silver.propiedades
WHERE tipo_operacion IS NOT NULL AND moneda IS NOT NULL
ORDER BY tipo_operacion, moneda;

In [0]:
%sql

select *  from bootcamp.gold.dim_tipo_operacion 

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_tiempo;

CREATE TABLE bootcamp.gold.dim_tiempo
(
fecha_id BIGINT,
fecha date not null,
anio INT,
mes INT,
trimestre INT,
dia_semana string,
es_fin_de_semana BOOLEAN,
_created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT "Tabla Dimension Tiempo - Star Schema";

INSERT INTO bootcamp.gold.dim_tiempo (fecha_id, fecha, anio, mes, trimestre, dia_semana, es_fin_de_semana)
SELECT 
distinct 
cast(date_format(fecha_publicacion, "yyyyMmdd") as bigint) as fecha_id,
fecha_publicacion as fecha,
year(fecha_publicacion) as anio,
month(fecha_publicacion) as mes,
quarter (fecha_publicacion) as trimestre,
case dayofweek(fecha_publicacion)
    when 1 then "Domingo"
    when 2 then "Lunes"
    when 3 then "Martes"
    when 4 then "Miercoles"
    when 5 then "Jueves"
    when 6 then "Viernes"
    when 7 then "Sabado"
    end as dia_semana,
DAYOFWEEK(fecha_publicacion) IN (1, 7) as es_fin_de_semana
FROM bootcamp.silver.propiedades
WHERE fecha_publicacion IS NOT NULL;

In [0]:
%sql
select * from bootcamp.gold.dim_tiempo

In [0]:
%sql

drop table if exists  bootcamp.gold.dim_caracteristica;

create table  bootcamp.gold.dim_caracteristica (

    caracteristicas_id bigint GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    estado string,
    cochera boolean,
    _created_at timestamp default current_timestamp()
)
using delta
tblproperties (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT "Tabla Dimension Caracteristicas - Star Schema";

insert into bootcamp.gold.dim_caracteristica 
(estado,cochera)
select distinct
    COALESCE(estado, 'sin especificar') as estado,
    COALESCE(cochera, false) as cochera
FROM bootcamp.silver.propiedades
ORDER BY estado, cochera;

In [0]:
%sql

select *  from bootcamp.gold.dim_caracteristica

In [0]:
%sql
-- dim_orientacion
DROP TABLE IF EXISTS bootcamp.gold.dim_orientacion;

CREATE TABLE bootcamp.gold.dim_orientacion (
    orientacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    orientacion STRING NOT NULL,
    tipo_orientacion STRING,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimensión de orientación - Star Schema';

INSERT INTO bootcamp.gold.dim_orientacion (orientacion, tipo_orientacion)
SELECT DISTINCT
    COALESCE(orientacion, 'sin especificar') as orientacion,
    CASE 
        WHEN LOWER(orientacion) LIKE '%norte%' THEN 'norte'
        WHEN LOWER(orientacion) LIKE '%sur%' THEN 'sur'
        WHEN LOWER(orientacion) LIKE '%oeste%' THEN 'oeste'
        WHEN LOWER(orientacion) LIKE '%este%' THEN 'este'
        ELSE 'sin especificar'
    END as tipo_orientacion
FROM bootcamp.silver.propiedades
ORDER BY orientacion;

In [0]:
%sql

select * from bootcamp.gold.dim_orientacion

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.gold.fact_propiedades;

CREATE TABLE bootcamp.gold.fact_propiedades (
    row_hash STRING NOT NULL,
    zona_id BIGINT,
    tipo_operacion_id BIGINT,
    fecha_id BIGINT,
    caracteristicas_id BIGINT,
    orientacion_id BIGINT,
    precio DECIMAL(15,2),
    expensas DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    ambientes INT,
    url STRING,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Tabla de hechos de propiedades - Star Schema. PK = row_hash (MD5 de url + precio)';

INSERT INTO bootcamp.gold.fact_propiedades (
    row_hash, zona_id, tipo_operacion_id, fecha_id, caracteristicas_id, orientacion_id,
    precio, expensas, precio_por_m2,
    metros_cuadrados_totales, metros_cuadrados_cubiertos, ambientes, url
)
SELECT 
    MD5(CONCAT_WS('|', sp.url, CAST(sp.precio AS STRING))) AS row_hash,
    dz.zona_id,
    dt.tipo_operacion_id,
    dtf.fecha_id,
    dc.caracteristicas_id,
    do.orientacion_id,
    sp.precio, sp.expensas, sp.precio_por_m2,
    sp.metros_cuadrados_totales, sp.metros_cuadrados_cubiertos,
    sp.ambientes, sp.url
FROM bootcamp.silver.propiedades sp
LEFT JOIN bootcamp.gold.dim_zona dz ON sp.partido = dz.partido AND sp.region = dz.region
LEFT JOIN bootcamp.gold.dim_tipo_operacion dt ON sp.tipo_operacion = dt.tipo_operacion AND sp.moneda = dt.moneda
LEFT JOIN bootcamp.gold.dim_tiempo dtf ON sp.fecha_publicacion = dtf.fecha
LEFT JOIN bootcamp.gold.dim_caracteristicas dc ON COALESCE(sp.estado, 'sin especificar') = dc.estado AND COALESCE(sp.cochera, false) = dc.cochera
LEFT JOIN bootcamp.gold.dim_orientacion do ON COALESCE(sp.orientacion, 'sin especificar') = do.orientacion
LIMIT 10000;

SELECT COUNT(*) AS total_fact, COUNT(DISTINCT row_hash) AS hashes_unicos
FROM bootcamp.gold.fact_propiedades;

In [0]:
%sql
-- Nivel 1: Provincia
DROP TABLE IF EXISTS bootcamp.gold.dim_provincia_sf;

CREATE TABLE bootcamp.gold.dim_provincia_sf (
    provincia_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    provincia STRING NOT NULL,
    pais STRING DEFAULT 'Argentina',
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de provincias - Snowflake Schema';

INSERT INTO bootcamp.gold.dim_provincia_sf (provincia, pais)
SELECT DISTINCT 'Buenos Aires' as provincia, 'Argentina' as pais;

-- Nivel 2: Ciudad
DROP TABLE IF EXISTS bootcamp.gold.dim_ciudad_sf;

CREATE TABLE bootcamp.gold.dim_ciudad_sf (
    ciudad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    ciudad STRING NOT NULL,
    provincia_id BIGINT,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de ciudades - Snowflake Schema';

INSERT INTO bootcamp.gold.dim_ciudad_sf (ciudad, provincia_id)
SELECT DISTINCT
    CASE 
        WHEN sp.region = 'capital federal' THEN 'CABA'
        ELSE 'GBA'
    END as ciudad,
    1 as provincia_id
FROM bootcamp.silver.propiedades sp
WHERE sp.region IS NOT NULL;

-- Nivel 3: Zona (referencia a ciudad)
DROP TABLE IF EXISTS bootcamp.gold.dim_zona_sf;

CREATE TABLE bootcamp.gold.dim_zona_sf (
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    partido STRING NOT NULL,
    region STRING NOT NULL,
    ciudad_id BIGINT,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimensión de zonas - Snowflake Schema (normalizada)';

INSERT INTO bootcamp.gold.dim_zona_sf (partido, region, ciudad_id)
SELECT DISTINCT
    sp.partido,
    sp.region,
    dc.ciudad_id
FROM bootcamp.silver.propiedades sp
JOIN bootcamp.gold.dim_ciudad_sf dc ON 
    CASE 
        WHEN sp.region = 'capital federal' THEN 'CABA'
        ELSE 'GBA'
    END = dc.ciudad
WHERE sp.partido IS NOT NULL
ORDER BY sp.partido;

-- Query de ejemplo Snowflake (para que veas la complejidad de joins para solamente reconstruir una dimensión equivalena a la de STAR)
SELECT 
    dp.provincia,
    dc.ciudad,
    dz.partido, dz.region
FROM bootcamp.silver.propiedades sp
JOIN bootcamp.gold.dim_zona_sf dz ON sp.partido = dz.partido AND sp.region = dz.region
JOIN bootcamp.gold.dim_ciudad_sf dc ON dz.ciudad_id = dc.ciudad_id
JOIN bootcamp.gold.dim_provincia_sf dp ON dc.provincia_id = dp.provincia_id
LIMIT 20;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.gold.obt_propiedades_completa;

CREATE TABLE bootcamp.gold.obt_propiedades_completa (
    propiedad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    precio DECIMAL(15,2),
    moneda STRING,
    expensas DECIMAL(15,2),
    ambientes INT,
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    antiguedad INT,
    cochera BOOLEAN,
    precio_por_m2 DECIMAL(15,2),
    partido STRING,
    region STRING,
    ciudad STRING,
    provincia STRING DEFAULT 'Buenos Aires',
    pais STRING DEFAULT 'Argentina',
    tipo_operacion STRING,
    categoria_operacion STRING,
    estado STRING,
    categoria_estado STRING,
    orientacion STRING,
    tipo_orientacion STRING,
    fecha_publicacion DATE,
    anio INT,
    mes INT,
    trimestre INT,
    dia_semana STRING,
    es_fin_de_semana BOOLEAN,
    segmento_precio STRING,
    rango_metros STRING,
    rango_ambientes STRING,
    tiene_expensas BOOLEAN,
    ratio_m2_cubiertos_totales DECIMAL(5,2),
    url STRING,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'One Big Table - Propiedades completas denormalizadas';

INSERT INTO bootcamp.gold.obt_propiedades_completa (
    precio, moneda, expensas, ambientes,
    metros_cuadrados_totales, metros_cuadrados_cubiertos,
    antiguedad, cochera, precio_por_m2,
    partido, region, ciudad, provincia, pais,
    tipo_operacion, categoria_operacion,
    estado, categoria_estado,
    orientacion, tipo_orientacion,
    fecha_publicacion, anio, mes, trimestre, dia_semana, es_fin_de_semana,
    segmento_precio, rango_metros, rango_ambientes,
    tiene_expensas, ratio_m2_cubiertos_totales,
    url
)
SELECT 
    sp.precio, sp.moneda, sp.expensas, sp.ambientes,
    sp.metros_cuadrados_totales, sp.metros_cuadrados_cubiertos,
    sp.antiguedad, sp.cochera, sp.precio_por_m2,
    sp.partido,
    sp.region,
    CASE WHEN sp.region = 'capital federal' THEN 'CABA' ELSE 'GBA' END as ciudad,
    'Buenos Aires' as provincia, 'Argentina' as pais,
    sp.tipo_operacion,
    CASE 
        WHEN sp.tipo_operacion = 'alquiler' THEN 'residencial'
        WHEN sp.tipo_operacion = 'venta' THEN 'residencial'
        WHEN sp.tipo_operacion = 'alquiler_temporario' THEN 'temporal'
        ELSE 'otro'
    END as categoria_operacion,
    sp.estado,
    CASE 
        WHEN LOWER(sp.estado) LIKE '%nuevo%' THEN 'nuevo'
        WHEN LOWER(sp.estado) LIKE '%usado%' THEN 'usado'
        WHEN LOWER(sp.estado) LIKE '%remodelado%' THEN 'remodelado'
        ELSE 'sin especificar'
    END as categoria_estado,
    COALESCE(sp.orientacion, 'sin especificar') as orientacion,
    CASE 
        WHEN LOWER(sp.orientacion) LIKE '%norte%' THEN 'norte'
        WHEN LOWER(sp.orientacion) LIKE '%sur%' THEN 'sur'
        WHEN LOWER(sp.orientacion) LIKE '%este%' THEN 'este'
        WHEN LOWER(sp.orientacion) LIKE '%oeste%' THEN 'oeste'
        ELSE 'sin especificar'
    END as tipo_orientacion,
    sp.fecha_publicacion,
    YEAR(sp.fecha_publicacion) as anio, MONTH(sp.fecha_publicacion) as mes,
    QUARTER(sp.fecha_publicacion) as trimestre,
    CASE DAYOFWEEK(sp.fecha_publicacion)
        WHEN 1 THEN 'Domingo' WHEN 2 THEN 'Lunes' WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miércoles' WHEN 5 THEN 'Jueves' WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sábado'
    END as dia_semana,
    DAYOFWEEK(sp.fecha_publicacion) IN (1, 7) as es_fin_de_semana,
    CASE 
        WHEN sp.moneda = 'ARS' AND sp.precio < 300000 THEN 'bajo'
        WHEN sp.moneda = 'ARS' AND sp.precio < 500000 THEN 'medio'
        WHEN sp.moneda = 'ARS' AND sp.precio < 800000 THEN 'alto'
        WHEN sp.moneda = 'ARS' THEN 'premium'
        WHEN sp.moneda = 'USD' AND sp.precio < 500 THEN 'bajo'
        WHEN sp.moneda = 'USD' AND sp.precio < 1000 THEN 'medio'
        WHEN sp.moneda = 'USD' AND sp.precio < 2000 THEN 'alto'
        WHEN sp.moneda = 'USD' THEN 'premium'
        ELSE 'sin clasificar'
    END as segmento_precio,
    CASE 
        WHEN sp.metros_cuadrados_totales < 50 THEN 'pequeño'
        WHEN sp.metros_cuadrados_totales < 100 THEN 'mediano'
        WHEN sp.metros_cuadrados_totales < 200 THEN 'grande'
        ELSE 'muy grande'
    END as rango_metros,
    CASE 
        WHEN sp.ambientes = 1 THEN 'mono'
        WHEN sp.ambientes BETWEEN 2 AND 3 THEN '2-3'
        WHEN sp.ambientes >= 4 THEN '4+'
        ELSE 'sin especificar'
    END as rango_ambientes,
    sp.expensas IS NOT NULL AND sp.expensas > 0 as tiene_expensas,
    ROUND(sp.metros_cuadrados_cubiertos / NULLIF(sp.metros_cuadrados_totales, 0), 2) as ratio_m2_cubiertos_totales,
    sp.url
FROM bootcamp.silver.propiedades sp
LIMIT 10000;

SELECT COUNT(*) AS total_obt FROM bootcamp.gold.obt_propiedades_completa;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.gold.fact_consultas;

CREATE TABLE bootcamp.gold.fact_consultas (
    consulta_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    zona_id BIGINT,
    fecha_id BIGINT,
    propiedad_url STRING,
    tipo_consulta STRING,
    cantidad_consultas INT DEFAULT 1,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de hechos de consultas - Galaxy Schema (mockeada)';

INSERT INTO bootcamp.gold.fact_consultas (
    zona_id, fecha_id, propiedad_url, tipo_consulta, cantidad_consultas
)
SELECT 
    dz.zona_id,
    dtf.fecha_id,
    sp.url as propiedad_url,
    CASE (RANDOM() * 4)::INT
        WHEN 0 THEN 'vista'
        WHEN 1 THEN 'contacto'
        WHEN 2 THEN 'favorito'
        ELSE 'compartir'
    END as tipo_consulta,
    (RANDOM() * 10 + 1)::INT as cantidad_consultas
FROM bootcamp.silver.propiedades sp
JOIN bootcamp.gold.dim_zona dz ON sp.partido = dz.partido AND sp.region = dz.region
JOIN bootcamp.gold.dim_tiempo dtf ON sp.fecha_publicacion = dtf.fecha
WHERE sp.precio_por_m2 < (
    SELECT PERCENTILE_APPROX(precio_por_m2, 0.5) 
    FROM bootcamp.silver.propiedades
)
LIMIT 5000;


In [0]:
%sql
-- Análisis cruzado entre hechos
SELECT 
    dz.partido, dz.region,
    COUNT(DISTINCT fp.row_hash) as total_propiedades,
    COUNT(DISTINCT fc.consulta_id) as total_consultas,
    SUM(fc.cantidad_consultas) as cantidad_total_consultas,
    ROUND(AVG(fp.precio_por_m2), 2) as precio_m2_promedio,
    ROUND(SUM(fc.cantidad_consultas) * 1.0 / NULLIF(COUNT(DISTINCT fp.row_hash), 0), 2) as consultas_por_propiedad
FROM bootcamp.gold.dim_zona dz
LEFT JOIN bootcamp.gold.fact_propiedades fp ON dz.zona_id = fp.zona_id
LEFT JOIN bootcamp.gold.fact_consultas fc ON dz.zona_id = fc.zona_id
GROUP BY dz.partido, dz.region
HAVING COUNT(DISTINCT fp.row_hash) > 0
ORDER BY consultas_por_propiedad DESC
LIMIT 20;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.silver.propiedades_into;

CREATE TABLE bootcamp.silver.propiedades_into (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para INSERT INTO';

-- Primera carga: alquileres
INSERT INTO bootcamp.silver.propiedades_into
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 500;

SELECT COUNT(*) as total_despues_primera_carga FROM bootcamp.silver.propiedades_into;

In [0]:
%sql
-- Segunda carga: ventas (INSERT INTO acumula)
INSERT INTO bootcamp.silver.propiedades_into
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'venta'
LIMIT 500;

SELECT tipo_operacion, COUNT(*) as cantidad
FROM bootcamp.silver.propiedades_into
GROUP BY tipo_operacion;

In [0]:
%sql
INSERT INTO bootcamp.silver.propiedades_into
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades sp
WHERE tipo_operacion = 'alquiler_temporario'
    AND NOT EXISTS (
        SELECT 1 
        FROM bootcamp.silver.propiedades_into pi
        WHERE pi.url = sp.url
            AND pi.precio = sp.precio
    )
LIMIT 300;

SELECT COUNT(*) as total_registros FROM bootcamp.silver.propiedades_into;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.silver.propiedades_overwrite;

CREATE TABLE bootcamp.silver.propiedades_overwrite (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para INSERT INTO';


-- Primera carga
INSERT OVERWRITE bootcamp.silver.propiedades_overwrite
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 1000;

SELECT COUNT(*) as total_registros FROM bootcamp.silver.propiedades_overwrite;

In [0]:
%sql
-- Segunda carga: OVERWRITE reemplaza todo
INSERT OVERWRITE bootcamp.silver.propiedades_overwrite
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'venta'
LIMIT 1000;

-- Solo tiene 'venta' (no acumuló 'alquiler')
SELECT tipo_operacion, COUNT(*) as cantidad
FROM bootcamp.silver.propiedades_overwrite
GROUP BY tipo_operacion;

In [0]:
%sql
SELECT 
    *,
    MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash
FROM bootcamp.silver.propiedades
LIMIT 5;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.silver.propiedades_merge_insert;

CREATE TABLE bootcamp.silver.propiedades_merge_insert (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING PRIMARY KEY,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE (solo INSERT)';

INSERT INTO bootcamp.silver.propiedades_merge_insert
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 100;

SELECT COUNT(*) as total_inicial FROM bootcamp.silver.propiedades_merge_insert;

In [0]:
%sql
MERGE INTO bootcamp.silver.propiedades_merge_insert AS target
USING (
    SELECT partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion IN ('alquiler', 'venta')
    LIMIT 200
) AS source
ON target.url = source.url
WHEN NOT MATCHED THEN
    INSERT (partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url, _processing_timestamp)
    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
        source.metros_cuadrados_totales, source.precio_por_m2, source.url, CURRENT_TIMESTAMP());

SELECT tipo_operacion, COUNT(*) as cantidad
FROM bootcamp.silver.propiedades_merge_insert
GROUP BY tipo_operacion;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.silver.propiedades_merge_upsert;

CREATE TABLE bootcamp.silver.propiedades_merge_upsert (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE (UPSERT)';

INSERT INTO bootcamp.silver.propiedades_merge_upsert
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp,
    CURRENT_TIMESTAMP() as _updated_at
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 100;

SELECT COUNT(*) as total_inicial FROM bootcamp.silver.propiedades_merge_upsert;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW datos_actualizados AS
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url
FROM (
    SELECT partido, region, tipo_operacion,
        precio * 1.10 as precio, moneda,
        metros_cuadrados_totales,
        precio_por_m2 * 1.10 as precio_por_m2, url,
        ROW_NUMBER() OVER (PARTITION BY precio, url ORDER BY precio) as rn
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion = 'alquiler'
) a
WHERE rn = 1
LIMIT 150;

MERGE INTO bootcamp.silver.propiedades_merge_upsert AS target
USING datos_actualizados AS source
ON target.url = source.url AND target.precio = source.precio
WHEN MATCHED THEN
    UPDATE SET
        precio_por_m2 = source.precio_por_m2,
        _updated_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
    INSERT (partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        _processing_timestamp, _updated_at)
    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
        source.metros_cuadrados_totales, source.precio_por_m2, source.url,
        CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP());

SELECT COUNT(*) as total_despues_upsert FROM bootcamp.silver.propiedades_merge_upsert;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.silver.propiedades_merge_hash;

CREATE TABLE bootcamp.silver.propiedades_merge_hash (
    row_hash STRING NOT NULL,
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE con Hash';

INSERT INTO bootcamp.silver.propiedades_merge_hash
SELECT 
    MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash,
    partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP() as _processing_timestamp
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 200;

SELECT COUNT(*) as total_inicial FROM bootcamp.silver.propiedades_merge_hash;

In [0]:
%sql
MERGE INTO bootcamp.silver.propiedades_merge_hash AS target
USING (
    SELECT 
        MD5(CONCAT_WS('|', url, CAST(precio AS STRING))) AS row_hash,
        partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion IN ('alquiler', 'venta')
    LIMIT 400
) AS source
ON target.row_hash = source.row_hash
WHEN NOT MATCHED THEN
    INSERT (row_hash, partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url, _processing_timestamp)
    VALUES (source.row_hash, source.partido, source.region, source.tipo_operacion, source.precio,
        source.moneda, source.metros_cuadrados_totales, source.precio_por_m2,
        source.url, CURRENT_TIMESTAMP());

SELECT COUNT(*) as total_despues_merge, COUNT(DISTINCT row_hash) as hashes_unicos
FROM bootcamp.silver.propiedades_merge_hash;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.silver.propiedades_merge_delete;

CREATE TABLE bootcamp.silver.propiedades_merge_delete (
    partido STRING,
    region STRING,
    tipo_operacion STRING,
    precio DECIMAL(15,2),
    moneda STRING,
    metros_cuadrados_totales DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    url STRING,
    _processing_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    _is_deleted BOOLEAN DEFAULT false
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de prueba para MERGE con soft delete';

INSERT INTO bootcamp.silver.propiedades_merge_delete
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url,
    CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), false
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 100;

SELECT COUNT(*) as total_inicial,
    SUM(CASE WHEN _is_deleted THEN 1 ELSE 0 END) as borrados
FROM bootcamp.silver.propiedades_merge_delete;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW datos_nuevos_delete AS
SELECT partido, region, tipo_operacion, precio, moneda,
    metros_cuadrados_totales, precio_por_m2, url
FROM bootcamp.silver.propiedades
WHERE tipo_operacion = 'alquiler'
LIMIT 50;

MERGE INTO bootcamp.silver.propiedades_merge_delete AS target
USING datos_nuevos_delete AS source
ON target.url = source.url
WHEN MATCHED THEN
    UPDATE SET
        precio = source.precio,
        precio_por_m2 = source.precio_por_m2,
        _updated_at = CURRENT_TIMESTAMP(),
        _is_deleted = false
WHEN NOT MATCHED THEN
    INSERT (partido, region, tipo_operacion, precio, moneda,
        metros_cuadrados_totales, precio_por_m2, url,
        _processing_timestamp, _updated_at, _is_deleted)
    VALUES (source.partido, source.region, source.tipo_operacion, source.precio, source.moneda,
        source.metros_cuadrados_totales, source.precio_por_m2, source.url,
        CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), false)
WHEN NOT MATCHED BY SOURCE THEN
    UPDATE SET _is_deleted = true, _updated_at = CURRENT_TIMESTAMP();

SELECT 
    COUNT(*) as total,
    SUM(CASE WHEN _is_deleted THEN 1 ELSE 0 END) as soft_deleted,
    SUM(CASE WHEN NOT _is_deleted THEN 1 ELSE 0 END) as activos
FROM bootcamp.silver.propiedades_merge_delete;